![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 30 -- Lab 1: Three Ways to Train a Pretrained ResNet

The same pretrained ResNet-18, the same 4,317 Kaggle flower photos, the same validation split, three training strategies from today's lecture: the encoder frozen and only the head trained (feature extraction, linear probing); `layer4` thawed with its own small learning rate next to the head's normal one (layer freezing with two parameter groups); everything thawed with a small rate for the encoder. For each one you count the trainable parameters, time the epochs, read the validation curve, and at the end measure how far each stage's weights moved away from the ImageNet values.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `alxmamaev/flowers-recognition` (about 230 MB), downloaded by `kagglehub` in the setup cell: `flowers/<class>/*.jpg`, 4,317 photos in five folders; 864 photos (20%) are held out for validation, the same split for all three strategies. Use a GPU (Runtime, Change runtime type, T4): the nine epochs take about 10 minutes there, far longer on a CPU.

---
# Setup

In [ ]:
# kagglehub fetches the Kaggle dataset; Colab has everything else.
!pip install -q kagglehub

In [ ]:
import os, random, time, copy
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("alxmamaev/flowers-recognition")
ROOT = os.path.join(path, "flowers")
CLASSES = sorted(os.listdir(ROOT))
print(CLASSES, {c: len(os.listdir(os.path.join(ROOT, c))) for c in CLASSES})

---
# Part 1 -- The data, once for all three strategies

Day 29's Lab 2 pipeline, written for you: the ImageNet contract, `ImageFolder` twice, one shuffled split. Every strategy below sees exactly the same training and validation photos, so the differences you measure come from the strategy alone.

In [ ]:
normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])   # the ImageNet contract
train_tf = transforms.Compose([transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), normalize])
val_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                             transforms.ToTensor(), normalize])
full_train = datasets.ImageFolder(ROOT, transform=train_tf)
full_val = datasets.ImageFolder(ROOT, transform=val_tf)
idx = list(range(len(full_train)))
random.shuffle(idx)
val_idx, train_idx = idx[:864], idx[864:]
train_loader = DataLoader(Subset(full_train, train_idx), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(Subset(full_val, val_idx), batch_size=64, shuffle=False, num_workers=2)
print(len(train_idx), "train,", len(val_idx), "val,", len(train_loader), "batches per epoch")

## Task 1: The contract check

Take one batch `xb, yb = next(iter(train_loader))` and print `xb.shape`, `xb.mean()`, `xb.std()` and `yb[:8]`. The slide's check from yesterday: a shape of (32, 3, 224, 224), a mean near 0, a standard deviation near 1.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(xb.shape) == (32, 3, 224, 224) and abs(xb.mean()) < 1 and 0.5 < xb.std() < 1.6, 'a batch of (32, 3, 224, 224) with mean near 0 and std near 1: check the Normalize in train_tf'
print('Task 1 passed')

---
# Part 2 -- The tools: counting, one epoch, evaluating, measuring movement

The *Mechanism as a Picture* and *Math* slides. Trainable means `requires_grad` is `True` and the optimizer holds the parameter; the four moves are Week 3's; the movement of a stage is how far its weights ended from where the file put them.

## Task 2: Count what can move

Write `count_trainable(model)` that returns the number of parameters with `requires_grad` set to `True`, `sum(p.numel() for p in model.parameters() if p.requires_grad)`. Call it on a fresh `resnet18(weights=ResNet18_Weights.DEFAULT)`: nothing is frozen yet, so the answer is the whole model, 11,689,512.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert count_trainable(fresh) == 11_689_512, 'count_trainable: every parameter of a fresh resnet18 has requires_grad=True: 11,689,512'
print('Task 2 passed')

## Task 3: One epoch, the four moves

Write `train_one_epoch(model, loader, optimizer)`: put the model in `train()` mode, then for every batch move `xb, yb` to `device`, compute `loss = criterion(model(xb), yb)`, do the four moves (`optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`) and add `loss.item() * len(yb)` to a running total; return the mean training loss over the epoch. `criterion = nn.CrossEntropyLoss()` is created for you below. This is the loop of Day 29's Lab 2 with the epoch loop taken outside, so that each strategy can call it.

Syntax hint (the shape of the call, not the answer):

```python
def train_one_epoch(model, loader, optimizer):
    model.train()
    for xb, yb in loader:
        ...                      # the four moves
    return mean_loss
```

In [ ]:
# Your code here


In [ ]:
def evaluate(model, loader):
    """Mean loss and accuracy on a loader, no gradients (Day 29, Lab 2)."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total_loss += criterion(out, yb).item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n

STAGES = ["conv1", "layer1", "layer2", "layer3", "layer4", "fc"]

def snapshot(model):
    """A copy of every weight tensor, by stage, to measure movement later."""
    return {s: [p.detach().cpu().clone() for p in getattr(model, s).parameters()] for s in STAGES}

def movement(before, model):
    """Relative change of each stage: ||w_after - w_before|| / ||w_before||, over all its weights."""
    out = {}
    for s in STAGES:
        after = [p.detach().cpu() for p in getattr(model, s).parameters()]
        diff = torch.sqrt(sum(((a - b) ** 2).sum() for a, b in zip(after, before[s])))
        norm = torch.sqrt(sum((b ** 2).sum() for b in before[s]))
        out[s] = (diff / norm).item()
    return out

print("tools ready")

---
# Part 3 -- Strategy A: the encoder frozen, the head trained (linear probing)

The *Two Definitions* and *In Code: The Two Ways* slides, yesterday's recipe: freeze everything, a new `Linear(512, 5)`, an optimizer that holds only the head.

## Task 4: Build model A

Create `model_a = resnet18(weights=ResNet18_Weights.DEFAULT)`, freeze every parameter with a loop that sets `param.requires_grad = False`, replace the head with `nn.Linear(model_a.fc.in_features, len(CLASSES))`, move it to `device`, and create `optimizer_a = torch.optim.Adam(model_a.fc.parameters(), lr=1e-3)`. Print `count_trainable(model_a)`: the slide says 2,565.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert count_trainable(model_a) == 2565 and len(optimizer_a.param_groups) == 1, 'model A: 2,565 trainable parameters (the head only) and one parameter group'
print('Task 4 passed')

## Task 5: Train A for 3 epochs

Take `start_a = snapshot(model_a)` before training (the ImageNet values, plus the random head). Then write the epoch loop out: for 3 epochs, note the time, call `train_one_epoch(model_a, train_loader, optimizer_a)`, then `evaluate(model_a, val_loader)`, append `(train_loss, val_loss, val_acc, seconds)` to `history_a` and print the four numbers. Note the seconds per epoch: nothing flows backward through the encoder here.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(history_a) == 3 and max(h[2] for h in history_a) > 0.8, 'three epochs of the head alone should pass 0.8 validation accuracy on the flowers'
print('Task 5 passed')

---
# Part 4 -- Strategy B: `layer4` thawed, two learning rates

The *In Code: Thaw layer4* slide. Freeze everything, thaw the last stage, a new head, and an optimizer with two parameter groups: the pretrained stage takes small steps, the random head normal ones.

## Task 6: Build model B

Create `model_b` the same way as A (load, freeze all, new head, to `device`), but after the freeze loop set `param.requires_grad = True` for every parameter of `model_b.layer4`. Then create `optimizer_b = torch.optim.Adam([...])` with two dictionaries: `{"params": model_b.layer4.parameters(), "lr": 1e-4}` and `{"params": model_b.fc.parameters(), "lr": 1e-3}`. Print the trainable count (the slide says 8,396,293) and `[g["lr"] for g in optimizer_b.param_groups]`.

Syntax hint (the shape of the call, not the answer):

```python
optimizer = torch.optim.Adam([
    {"params": part1.parameters(), "lr": ...},
    {"params": part2.parameters(), "lr": ...},
])
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert count_trainable(model_b) == 8_396_293 and len(optimizer_b.param_groups) == 2, 'model B: layer4 + head = 8,396,293 trainable parameters, two parameter groups'
print('Task 6 passed')

## Task 7: Train B for 3 epochs

As in Task 5: `start_b = snapshot(model_b)`, then the written-out epoch loop for 3 epochs into `history_b`, printing the four numbers. Compare the seconds per epoch with A: the gradient now flows back to the start of `layer4`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(history_b) == 3 and max(h[2] for h in history_b) > 0.8, 'three epochs with layer4 thawed should pass 0.8 validation accuracy'
print('Task 7 passed')

---
# Part 5 -- Strategy C: everything thawed, small steps for the encoder

The *Math* slide: a `layer4` weight is about 0.010 in size, so the encoder gets `1e-5` and only the random head keeps `1e-3`. Nothing is frozen; the freeze loop disappears.

## Task 8: Build model C

Create `model_c = resnet18(weights=ResNet18_Weights.DEFAULT)` with no freeze loop, a new head, to `device`. Collect the encoder's parameters as `encoder_params = [p for n, p in model_c.named_parameters() if not n.startswith("fc")]`, and create `optimizer_c` with two groups: `encoder_params` at `1e-5` and `model_c.fc.parameters()` at `1e-3`. Print the trainable count: all 11,179,077.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert count_trainable(model_c) == 11_179_077 and len(optimizer_c.param_groups) == 2, 'model C: all 11,179,077 parameters trainable, two parameter groups'
print('Task 8 passed')

## Task 9: Train C for 3 epochs

`start_c = snapshot(model_c)`, then the same written-out loop for 3 epochs into `history_c`. The gradient now flows through all 11 million weights: watch the seconds.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(history_c) == 3 and max(h[2] for h in history_c) > 0.8, 'three epochs with everything thawed at 1e-5 (head 1e-3) should pass 0.8 validation accuracy'
print('Task 9 passed')

---
# Part 6 -- Compare

The *Evidence* and *Strategies* slides gave other people's numbers; these are yours, on one split.

## Task 10: The comparison table

Build a DataFrame `table` with one row per strategy (`A: head only`, `B: layer4 + head`, `C: everything`) and the columns `trainable` (from `count_trainable`), `best_val_acc` (the best of the three epochs), `final_train_loss` and `seconds_per_epoch` (the mean over the three epochs). Print it. Which strategy wins on accuracy, which on time, and how far apart are they?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert list(table['trainable']) == [2565, 8396293, 11179077] and table['best_val_acc'].min() > 0.8, 'table: trainable counts 2,565 / 8,396,293 / 11,179,077 and every strategy above 0.8'
print('Task 10 passed')

## Task 11: The three validation curves

Plot the validation accuracy against the epoch (1 to 3) for the three strategies on one axis, with a legend, axis labels and a title; add a dashed horizontal line at 0.68, Day 23's from-scratch scenario. Three epochs are few: is any curve still rising?

In [ ]:
# Your code here


## Task 12: How far did the weights move?

For each model compute `movement(start, model)` with its own snapshot, and put the three results in a DataFrame with the stages as rows and the strategies as columns (`pd.DataFrame({...}).round(4)`). Print it. Read it against the slide: a frozen stage moves exactly 0; the head moves the most everywhere (it started random); in C every stage moved, but by how much compared with `layer4` in B?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert moves.loc['layer4', 'A: head only'] == 0 and moves.loc['layer1', 'B: layer4 + head'] == 0 and moves.loc['layer4', 'B: layer4 + head'] > 0 and moves.loc['conv1', 'C: everything'] > 0, 'movement: frozen stages read exactly 0, thawed stages read above 0'
print('Task 12 passed')

---
## Task 13: Reflect

Task 10 put three numbers side by side: which strategy would you hand in on Friday if the budget were 10 minutes, and which if it were an hour? In Task 12 the head moved by far the most in every strategy: why is that expected, and why is it not a problem? Compare the movement of `layer4` in B (learning rate 1e-4) with `layer4` in C (1e-5): does the ratio match what the *Math* slide predicted, and what would you expect at 1e-3? Strategy C left `conv1` almost where ImageNet put it even though nothing was frozen: give the two reasons from the lecture. Finally, in what order would you run the three strategies in a real competition, and what number decides whether you go on to the next one?

*Your answers here*

---
## Conclusion

- **One split, three strategies**: the same pretrained ResNet-18 trained with the encoder frozen, with `layer4` thawed, and with everything thawed, each with its own learning rates per parameter group.
- **Trainable counts and time**: 2,565 against 8,396,293 against 11,179,077 parameters, the seconds per epoch measured for each, the validation curves on one axis.
- **Parameter groups in code**: an optimizer built from a list of `{"params": ..., "lr": ...}` dictionaries, small steps for pretrained stages and normal steps for the random head.
- **Movement measured**: frozen stages at exactly 0, the head far ahead, and the learning rate deciding how far a thawed stage walks from its ImageNet values.

---

Made By **Sattam Altwaim**